In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.layers import Dense, Dropout, Flatten,LSTM,GRU
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential

In [2]:
clns=["unit_number","time_cycles","op_setting_1","op_setting_2","op_setting_3"]+[f"sensor{i}" for i in range(1,22)]
fe=["op_setting_1","op_setting_2","op_setting_3"]+[f"sensor{i}" for i in range(1, 22)]

print(len(clns))

26


In [3]:
train_1=pd.read_csv("./nasa/train_FD001.txt",sep=r"\s+",header=None,names=clns)

In [4]:
test_1=pd.read_csv("./nasa/test_FD001.txt",sep=r"\s+",header=None,names=clns)

In [5]:
rul_1=pd.read_csv("./nasa/rul_FD001.txt",sep=r"\s+",header=None,names=["rul"])

In [6]:
mx_c=train_1.groupby("unit_number")["time_cycles"].transform("max")
train_1["rul"]=mx_c-train_1["time_cycles"]

In [7]:
sc=MinMaxScaler()
train_1[fe]=sc.fit_transform(train_1[fe])
test_1[fe]=sc.transform(test_1[fe])

In [8]:
s_l=30
xl=[]
yl=[]
for i in train_1["unit_number"].unique():
    en_data=train_1[train_1["unit_number"]==i].sort_values("time_cycles")
    data=en_data[fe].values
    rul_val=en_data["rul"].values

    for j in range(0,len(data)-s_l+1):
        wi=data[j:j+s_l]
        tar=rul_val[j+s_l-1]
        xl.append(wi)
        yl.append(tar)
x_train=np.array(xl)
y_train=np.array(yl)

In [9]:
print(len(fe))

24


In [10]:
model=Sequential()
model.add(GRU(64,return_sequences=True,input_shape=(s_l,24)))
model.add(Dropout(0.4))
model.add(GRU(16,return_sequences=False))
model.add(Dense(1))

C:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [11]:
compile=model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 30, 64)         │        17,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 16)             │         3,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,233 (82.94 KB)

 Trainable params: 21,233 (82.94 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
spl=int(len(x_train)*0.8)

xval=x_train[spl:]
yval=y_train[:spl]

x_train=x_train[:spl]
y_train=y_train[:spl]

In [13]:
history=model.fit(x_train,y_train,validation_split=0.2,epochs=100,batch_size=32)

Epoch 1/100
444/444 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 10384.8818 - mae: 83.7063 - val_loss: 14084.4766 - val_mae: 95.1000
Epoch 2/100
444/444 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 9173.6387 - mae: 77.1242 - val_loss: 12788.3779 - val_mae: 89.1788
Epoch 3/100
444/444 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 8166.2534 - mae: 71.7027 - val_loss: 11647.0391 - val_mae: 84.0399
Epoch 4/100
444/444 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 7294.5806 - mae: 67.0310 - val_loss: 10640.8096 - val_mae: 79.5818
Epoch 5/100
444/444 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 6540.8911 - mae: 63.0389 - val_loss: 9752.7734 - val_mae: 75.7020
Epoch 6/100
444/444 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 5894.8105 - mae: 59.6350 - val_loss: 8974.6494 - val_mae: 72.3549
Epoch 7/100
444/444 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 5345.7173 - mae: 56.7811 - val_loss: 8298.7012 - val_mae: 69.5004
Epoch 8/100
444/444 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 4884.5630 - mae: 54.4276 - val_loss: 771